# 📘 Clase 20: Árboles de decisión y Random Forest

**Idea central:** Un Random Forest combina muchos árboles de decisión para dar respuestas más confiables que cualquier árbol solo.

En esta clase clasificaremos estudiantes como **Aprobado** o **En Riesgo** usando árboles de decisión. Luego veremos cómo el Random Forest supera al árbol individual.

## 📦 Celda 1 — Importar librerías y preparar datos

In [ ]:
# Qué hace: cargar el dataset de estudiantes y crear la etiqueta de clasificación.
# Para qué sirve: tener un problema real de clasificación binaria para entrenar los árboles.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../datasets/estudiantes.csv")

# Etiqueta: 1 = Aprobado, 0 = En Riesgo
df["estado"] = ((df["evaluacion_python"] >= 60) & (df["asistencia_pct"] >= 70)).astype(int)

print("Distribución de la etiqueta:")
print(df["estado"].value_counts().rename({0: "En Riesgo", 1: "Aprobado"}))
df.head()

## ✂️ Celda 2 — Separar variables y dividir datos

In [ ]:
# Qué hace: definir las variables predictoras y dividir en train/test.
# Para qué sirve: tener conjuntos separados para entrenar y evaluar honestamente.
X = df[["asistencia_pct", "evaluacion_python", "evaluacion_pandas", "edad"]]
y = df["estado"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} filas  |  Test: {X_test.shape[0]} filas")

## 🌳 Celda 3 — Árbol de decisión con max_depth=3

Empezamos con un árbol poco profundo: generaliza bien aunque no memorice todos los datos.

In [ ]:
# Qué hace: entrenar un DecisionTreeClassifier con profundidad limitada.
# Para qué sirve: tener un modelo interpretable como punto de partida.
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)

acc_train = arbol.score(X_train, y_train)
acc_test  = arbol.score(X_test,  y_test)

print(f"Árbol (max_depth=3)")
print(f"  Accuracy en TRAIN: {acc_train:.3f}")
print(f"  Accuracy en TEST:  {acc_test:.3f}")

## 🖼️ Celda 4 — Visualizar el árbol

In [ ]:
# Qué hace: graficar el árbol de decisión con colores por clase.
# Para qué sirve: ver exactamente qué preguntas hace el modelo y en qué orden.
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(
    arbol,
    feature_names=X.columns.tolist(),
    class_names=["En Riesgo", "Aprobado"],
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=9
)
plt.title("Árbol de decisión — Clase 20", fontsize=13)
plt.tight_layout()
plt.show()

## ⚠️ Celda 5 — Sobreajuste: árbol sin límite de profundidad

In [ ]:
# Qué hace: entrenar árboles con distintas profundidades y comparar train vs test.
# Para qué sirve: mostrar visualmente el problema del sobreajuste.
profundidades = [1, 2, 3, 5, 10, None]
resultados = []

for d in profundidades:
    a = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    resultados.append({
        "max_depth": str(d),
        "Train": round(a.score(X_train, y_train), 3),
        "Test":  round(a.score(X_test,  y_test),  3)
    })

res_df = pd.DataFrame(resultados)
print(res_df.to_string(index=False))
print("\n⚠️ Cuando Train >> Test: el árbol memoriza en lugar de aprender (sobreajuste).")

## 🌲🌲🌲 Celda 6 — Random Forest: muchos árboles, una respuesta

In [ ]:
# Qué hace: entrenar un RandomForestClassifier con 100 árboles.
# Para qué sirve: mostrar cómo el bagging reduce la varianza y mejora el accuracy en test.
bosque = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
bosque.fit(X_train, y_train)

print("Random Forest (100 árboles, max_depth=5)")
print(f"  Accuracy en TRAIN: {bosque.score(X_train, y_train):.3f}")
print(f"  Accuracy en TEST:  {bosque.score(X_test,  y_test):.3f}")

## 📊 Celda 7 — Importancia de variables

In [ ]:
# Qué hace: graficar la importancia de cada variable según el Random Forest.
# Para qué sirve: comunicar de forma clara cuál información es más valiosa para el modelo.
importancias = pd.Series(bosque.feature_importances_, index=X.columns).sort_values()

importancias.plot(kind="barh", color="forestgreen", figsize=(7, 4))
plt.title("Importancia de variables — Random Forest")
plt.xlabel("Importancia relativa (suma = 1)")
plt.tight_layout()
plt.show()

print("\nTop variable más importante:", importancias.idxmax())

## 🔮 Celda 8 — Predecir estudiantes nuevos con probabilidades

In [ ]:
# Qué hace: crear estudiantes hipotéticos y predecir su estado con probabilidades.
# Para qué sirve: practicar inferencia real con predict_proba para ver la confianza del modelo.
nuevos = pd.DataFrame({
    "asistencia_pct":    [95,  50,  75],
    "evaluacion_python": [85,  45,  65],
    "evaluacion_pandas": [80,  40,  60],
    "edad":              [22,  19,  25]
})

clases  = bosque.predict(nuevos)
probas  = bosque.predict_proba(nuevos)

for i, (c, p) in enumerate(zip(clases, probas)):
    estado = "Aprobado" if c == 1 else "En Riesgo"
    print(f"Estudiante {i+1}: {estado}  (confianza: {max(p)*100:.1f}%)")

## 📋 Celda 9 — Comparación final: árbol vs bosque

In [ ]:
# Qué hace: mostrar en una tabla el accuracy test del árbol (max_depth=3) vs Random Forest.
# Para qué sirve: consolidar visualmente la ventaja del bosque frente al árbol individual.
comparacion = pd.DataFrame({
    "Modelo": ["Árbol (max_depth=3)", "Random Forest (100 árboles)"],
    "Accuracy en TEST": [
        round(arbol.score(X_test, y_test), 3),
        round(bosque.score(X_test, y_test), 3)
    ]
})
print(comparacion.to_string(index=False))
print("\n💡 El bosque vence al árbol porque promedia los errores de muchos árboles independientes.")

## 🔁 Celda 10 — Efecto del número de árboles (n_estimators)

In [ ]:
# Qué hace: graficar cómo cambia el accuracy al aumentar el número de árboles.
# Para qué sirve: mostrar que más árboles mejoran hasta cierto punto, luego estabilizan.
n_vals = [1, 5, 10, 25, 50, 100, 200]
accs   = []

for n in n_vals:
    rf = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    accs.append(rf.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(n_vals, accs, marker="o", color="forestgreen")
plt.xlabel("Número de árboles (n_estimators)")
plt.ylabel("Accuracy en TEST")
plt.title("¿Cuántos árboles necesitamos?")
plt.tight_layout()
plt.show()
print("Observación: el accuracy se estabiliza a partir de cierto número de árboles.")